In [24]:
from sentence_transformers import SentenceTransformer
import json

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9474.83it/s]


In [25]:
import pandas as pd

with open("dataset/train_data/task1_corpus.json", "r") as f:
    data = json.load(f)

with open("dataset/train_data/task1_train_queries.json", "r") as f:
    train_data = json.load(f)

with open("dataset/task1_test_queries.json", "r") as f:
    test_data = json.load(f)

db = pd.DataFrame(data)
train = pd.DataFrame(train_data)
test = pd.DataFrame(test_data)

print (train.head())
print ("=" * 30)
print (test.head())
print ("=" * 30)
print (db.head())

        query_id query_language  \
0  q_train_00000             es   
1  q_train_00001             es   
2  q_train_00002             es   
3  q_train_00003             es   
4  q_train_00004             es   

                                          query_text correct_doc_id  
0                                   ¿Sois de Urumqi?      doc_01391  
1               Hay una televisión en mi habitación.      doc_00700  
2  Tom ha estado en el corredor de la muerte dura...      doc_01515  
3                                    No tengo ganas.      doc_01027  
4                           Daniela me llamó a casa.      doc_00439  
       query_id query_language                                      query_text
0  q_test_00000             es               Su tarjeta de crédito, por favor.
1  q_test_00001             es  Tienes que saber que estoy enamorado de María.
2  q_test_00002             es                  La guerra no le gusta a nadie.
3  q_test_00003             es                       

In [26]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers import SentenceTransformerTrainer

pairs = train.merge(
    db,
    left_on="correct_doc_id",
    right_on="doc_id",
    how="inner"
)

pairs = pairs[["query_text", "text"]]
pairs.columns = ["anchor", "positive"]

train_pairs, validation_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

dataset = Dataset.from_pandas(train_pairs, preserve_index=False)
print(dataset.column_names)

print(dataset[0])
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    output_dir="dataset/model",
    num_train_epochs=10,
    per_device_train_batch_size=32,
)
trainer = SentenceTransformerTrainer(
    model, args, dataset, loss=loss
)

trainer.train()


['anchor', 'positive']
{'anchor': 'A agencia nacional confirmou oficialmente a construcao da nova sede de NitroAssociation.', 'positive': 'Agentia Nationala a confirmat oficial constructia noului sediu al NitroAssociation.'}


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.004719
1000,0.001918


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.18it/s]
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.78it/s]
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


TrainOutput(global_step=1250, training_loss=0.0031169740676879882, metrics={'train_runtime': 552.8589, 'train_samples_per_second': 72.351, 'train_steps_per_second': 2.261, 'total_flos': 0.0, 'train_loss': 0.0031169740676879882, 'epoch': 10.0})

In [27]:
embeddings = model.encode(db["text"].tolist())
embeddings

array([[-0.380353  , -0.12877676,  0.38276073, ..., -0.24614263,
        -0.17651282,  0.0176382 ],
       [-0.29922533, -0.2027036 ,  0.40634936, ...,  0.6639217 ,
         0.4387903 ,  0.32988322],
       [ 0.22942139,  0.00377708, -0.25335196, ..., -0.05467197,
         0.23607397, -0.11792452],
       ...,
       [-0.07739267,  0.05874131,  0.1381369 , ..., -0.06425773,
         0.10638992, -0.02162638],
       [ 0.16185336,  0.15931548, -0.28899226, ..., -0.28015286,
        -0.13915253, -0.06801365],
       [ 0.30576593, -0.06811742, -0.25280714, ..., -0.1580414 ,
        -0.0155824 ,  0.10888245]], shape=(15000, 384), dtype=float32)

In [28]:
test.head()

,query_id,query_language,query_text
0,q_test_00000,es,"Su tarjeta de crédito, por favor."
1,q_test_00001,es,Tienes que saber que estoy enamorado de María.
2,q_test_00002,es,La guerra no le gusta a nadie.
3,q_test_00003,es,Me desnudo.
4,q_test_00004,pt,"""Alô."" ""Alô, quem fala?"""


In [29]:
submission = []

test_embeddings = model.encode(test["query_text"].tolist())

In [30]:
similarities = test_embeddings @ embeddings.T

print(test_embeddings.shape)
print(embeddings.T.shape)
print(similarities.shape)
print(type(similarities))

(1500, 384)
(384, 15000)
(1500, 15000)
<class 'numpy.ndarray'>


In [31]:
import numpy as np

best_indices = np.argmax(similarities, axis=1)

submission = [
    db.iloc[idx]["doc_id"]
    for idx in best_indices
]

final = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test["query_id"],
    "answer": submission
})

final.head()
final.to_csv("submission.csv", index=False)